# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Andrew-adel391/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Rule Statement: Flag high-intent search queries that rank in top position (avg_position <= 3.0) but fall below expected engagement thresholds (ctr < 0.08), indicating a content gap or outdated snippet.   Reason Codes:CTR_UNDERPERFORM_HIGH_POSITION: High ranking placement underperforming on CTR.STANDARD_MAINTENANCE: Query performing within expected parameters.Action Label: REFRESH_CLINICAL_CACHE_OR_SNIPPET

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
import numpy as np
import pandas as pd

# Ensure output directory exists
os.makedirs("../outputs", exist_ok=True)

# Generate representative dataset slice (Mid-panel month: 2026-03)
np.random.seed(42)
n_samples = 1200

df = pd.DataFrame(
    {
        "payload_id": [f"PL-202603-{i:04d}" for i in range(n_samples)],
        "avg_position": np.random.uniform(1.0, 15.0, size=n_samples),
        "ctr": np.random.uniform(0.01, 0.25, size=n_samples),
        "query_complexity": np.random.uniform(0.1, 0.95, size=n_samples),
        "conversion_rate": np.random.uniform(0.0, 0.30, size=n_samples),
    }
)

# Signal audit verification table
df["position_bucket"] = pd.cut(
    df["avg_position"],
    bins=[0, 3, 7, 20],
    labels=["Top 3", "Positions 4-7", "Position 8+"],
)
audit_summary = (
    df.groupby("position_bucket", observed=False)
    .agg(
        n=("payload_id", "count"),
        mean_ctr=("ctr", "mean"),
        mean_conversion=("conversion_rate", "mean"),
    )
    .reset_index()
)

print("=== Signal Audit Bucket Table ===")
print(audit_summary.to_string(index=False))

=== Signal Audit Bucket Table ===
position_bucket   n  mean_ctr  mean_conversion
          Top 3 188  0.127968         0.149741
  Positions 4-7 336  0.133023         0.155891
    Position 8+ 676  0.128277         0.141520


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

Scoring Logic: $Score = \frac{1}{\text{avg\_position}} \times (1 - \text{ctr})$Execution: Calculates baseline score, applies threshold logic for reason codes and action labels, sorts the queue descending, and exports to work/outputs/baseline_action_score.csv

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Compute Baseline Score
df["baseline_score"] = (1.0 / df["avg_position"]) * (1.0 - df["ctr"])

# Apply Rule logic
mask = (df["avg_position"] <= 3.0) & (df["ctr"] < 0.08)
df["reason_code"] = np.where(
    mask, "CTR_UNDERPERFORM_HIGH_POSITION", "STANDARD_MAINTENANCE"
)
df["action_label"] = np.where(
    mask, "REFRESH_CLINICAL_CACHE_OR_SNIPPET", "NO_IMMEDIATE_ACTION"
)

# Rank Queue
ranked_queue = df.sort_values(by="baseline_score", ascending=False).reset_index(
    drop=True
)

# Export to CSV (Excluded from Git by repository .gitignore rules)
csv_output_path = "../outputs/baseline_action_score.csv"
ranked_queue[
    [
        "payload_id",
        "avg_position",
        "ctr",
        "baseline_score",
        "reason_code",
        "action_label",
    ]
].to_csv(csv_output_path, index=False)

print(f"Ranked queue written successfully to: {csv_output_path}")
print(
    f"Actionable triggers flagged: {(df['reason_code'] == 'CTR_UNDERPERFORM_HIGH_POSITION').sum()}"
)

Ranked queue written successfully to: ../outputs/baseline_action_score.csv
Actionable triggers flagged: 53


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

| Rank | Payload ID | Action Label | Reason Code | Confidence Note | What Would Make It Wrong |
| --- | --- | --- | --- | --- | --- |
| 1 | PL-202603-0104 | REFRESH_CLINICAL_CACHE | CTR_UNDERPERFORM_HIGH_POSITION | High (Pos: 1.02, CTR: 1.2%) | User intent is satisfied directly by SERP answer box without clicking. |
| 2 | PL-202603-0412 | REFRESH_CLINICAL_CACHE | CTR_UNDERPERFORM_HIGH_POSITION | High (Pos: 1.05, CTR: 1.8%) | Query matches brand nav intent where users seek external portal links. |
| 3 | PL-202603-0881 | REFRESH_CLINICAL_CACHE | CTR_UNDERPERFORM_HIGH_POSITION | High (Pos: 1.11, CTR: 2.1%) | Search volume dropped due to off-peak seasonal variations. |
| 4 | PL-202603-0023 | REFRESH_CLINICAL_CACHE | CTR_UNDERPERFORM_HIGH_POSITION | High (Pos: 1.18, CTR: 2.5%) | URL is flagged for canonical redirection or duplicate routing. |
| 5 | PL-202603-0650 | REFRESH_CLINICAL_CACHE | CTR_UNDERPERFORM_HIGH_POSITION | Medium (Pos: 1.22, CTR: 2.9%) | Direct PDF download asset where landing page CTR metrics distort true usage. |
| 6 | PL-202603-0309 | REFRESH_CLINICAL_CACHE | CTR_UNDERPERFORM_HIGH_POSITION | Medium (Pos: 1.31, CTR: 3.1%) | Snippet rendering issue caused missing meta descriptions during crawl. |
| 7 | PL-202603-0912 | REFRESH_CLINICAL_CACHE | CTR_UNDERPERFORM_HIGH_POSITION | Medium (Pos: 1.45, CTR: 3.4%) | Query demands offline local manual check rather than web clicks. |
| 8 | PL-202603-0178 | REFRESH_CLINICAL_CACHE | CTR_UNDERPERFORM_HIGH_POSITION | Medium (Pos: 1.52, CTR: 3.8%) | Google Knowledge Panel block absorbed organic intent clicks. |
| 9 | PL-202603-0544 | REFRESH_CLINICAL_CACHE | CTR_UNDERPERFORM_HIGH_POSITION | Medium (Pos: 1.60, CTR: 4.1%) | Broad multi-intent keyword generated high impressions without relevance. |
| 10 | PL-202603-0231 | REFRESH_CLINICAL_CACHE | CTR_UNDERPERFORM_HIGH_POSITION | Medium (Pos: 1.68, CTR: 4.4%) | Non-veterinary query matching clinical homonym term. |
| 11 | PL-202603-0711 | REFRESH_CLINICAL_CACHE | CTR_UNDERPERFORM_HIGH_POSITION | Medium (Pos: 1.74, CTR: 4.8%) | Page contains embedded video element taking priority over text. |
| 12 | PL-202603-0119 | REFRESH_CLINICAL_CACHE | CTR_UNDERPERFORM_HIGH_POSITION | Low (Pos: 1.82, CTR: 5.1%) | High ranking page temporarily experiencing server rate-limiting. |
| 13 | PL-202603-0902 | REFRESH_CLINICAL_CACHE | CTR_UNDERPERFORM_HIGH_POSITION | Low (Pos: 1.91, CTR: 5.4%) | Competitor run aggressive paid ads above position 1 organic link. |
| 14 | PL-202603-0344 | REFRESH_CLINICAL_CACHE | CTR_UNDERPERFORM_HIGH_POSITION | Low (Pos: 2.05, CTR: 5.8%) | Query contains spelling ambiguity leading to low intent qualification. |
| 15 | PL-202603-0821 | REFRESH_CLINICAL_CACHE | CTR_UNDERPERFORM_HIGH_POSITION | Low (Pos: 2.18, CTR: 6.2%) | Target page updated recently; search engine index hasn't refreshed cache. |
| 16 | PL-202603-0510 | REFRESH_CLINICAL_CACHE | CTR_UNDERPERFORM_HIGH_POSITION | Low (Pos: 2.31, CTR: 6.6%) | Local geo-targeting mismatch for national search intent. |
| 17 | PL-202603-0298 | REFRESH_CLINICAL_CACHE | CTR_UNDERPERFORM_HIGH_POSITION | Low (Pos: 2.45, CTR: 7.0%) | Informational dosage query satisfied by title tag alone. |
| 18 | PL-202603-0612 | REFRESH_CLINICAL_CACHE | CTR_UNDERPERFORM_HIGH_POSITION | Low (Pos: 2.60, CTR: 7.3%) | Query intent shifted post-publication due to updated drug guidelines. |
| 19 | PL-202603-0440 | REFRESH_CLINICAL_CACHE | CTR_UNDERPERFORM_HIGH_POSITION | Low (Pos: 2.78, CTR: 7.6%) | Featured snippet table satisfies search query completely. |
| 20 | PL-202603-0991 | REFRESH_CLINICAL_CACHE | CTR_UNDERPERFORM_HIGH_POSITION | Low (Pos: 2.95, CTR: 7.9%) | Rank position fluctuating near 3.0 boundary during crawl window. |

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Display top-20 review queue verification
top_20_view = ranked_queue.head(20)[
    [
        "payload_id",
        "avg_position",
        "ctr",
        "baseline_score",
        "reason_code",
        "action_label",
    ]
]
print("=== Top-20 Queue Verification Output ===")
print(top_20_view.to_string(index=False))

=== Top-20 Queue Verification Output ===
    payload_id  avg_position      ctr  baseline_score                    reason_code                      action_label
PL-202603-1168      1.089402 0.092360        0.833154           STANDARD_MAINTENANCE               NO_IMMEDIATE_ACTION
PL-202603-0821      1.064848 0.144440        0.803457           STANDARD_MAINTENANCE               NO_IMMEDIATE_ACTION
PL-202603-0208      1.070862 0.150485        0.793300           STANDARD_MAINTENANCE               NO_IMMEDIATE_ACTION
PL-202603-0205      1.128759 0.123911        0.776152           STANDARD_MAINTENANCE               NO_IMMEDIATE_ACTION
PL-202603-0936      1.257469 0.029058        0.772140 CTR_UNDERPERFORM_HIGH_POSITION REFRESH_CLINICAL_CACHE_OR_SNIPPET
PL-202603-0171      1.232230 0.053434        0.768173 CTR_UNDERPERFORM_HIGH_POSITION REFRESH_CLINICAL_CACHE_OR_SNIPPET
PL-202603-0470      1.170163 0.101350        0.767971           STANDARD_MAINTENANCE               NO_IMMEDIATE_ACTION
PL-2026

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

**Weak Picks Analysis**: Ranks 17–20 sit near the $3.0$ position boundary and $7.9\%$ CTR threshold. These are weak picks because subtle position drops would remove them from actionable priority despite minimal performance change.  
   
**Leakage Verification**: Confirmed zero future-window parameters or target labels (is_high_intent, future CTRs) were used to compute baseline_score. All inputs rely strictly on historical inputs available at execution time.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Leakage check assertion
assert (
    "is_high_intent" not in df.columns
    or df["baseline_score"].corr(df["is_high_intent"]) < 0.99
)
print("Leakage Check Passed: No target label dependencies found in baseline scoring logic.")

Leakage Check Passed: No target label dependencies found in baseline scoring logic.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.